<a href="https://colab.research.google.com/github/MJMortensonWarwick/AnalyticsEngineering2026/blob/main/4_04_sql_cleaning_and_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 4.04 Cleaning and processing data with SQL (DuckDB)
The SQL companion to `3_02`: the same jobs, the same card-spend data, the same section order, written as SQL and run with [DuckDB](https://duckdb.org). DuckDB runs inside this notebook (no server to set up), reads CSV files directly, and is quick on tables of this size.

Why bother with both? In a warehouse, cleaning is done in SQL. Every dbt model is a SELECT statement, so the queries below are what you will write in dbt. Wherever pandas and SQL differ in a way that matters, there is a note.

**Sections**
1. Loading and first look
2. Selecting and filtering
3. Data types
4. Missing values
5. Duplicates
6. Text cleaning
7. Recoding and creating columns
8. Dates and times
9. Sorting and ranking
10. Grouping and aggregating
11. Window and running calculations
12. Reshaping
13. Combining tables
14. Validation and outliers
15. Chaining steps into a pipeline
16. Saving results, and moving between pandas and SQL
17. Quick reference (pandas next to SQL)

## 0. Setup

In [ ]:
%pip install -q duckdb

import glob, os, urllib.request, zipfile
import duckdb
import pandas as pd

print("duckdb", duckdb.__version__)

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/testgitsession/card-spend-practice.zip"
DATA_DIR = "/content/card-spend"

if not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/card-spend.zip")
    zipfile.ZipFile("/content/card-spend.zip").extractall(DATA_DIR)

def find(name):
    hits = glob.glob(os.path.join(DATA_DIR, "**", name), recursive=True)
    if not hits:
        raise FileNotFoundError(f"Missing {name}. Ask for help.")
    return hits[0]

con = duckdb.connect()          # an in-memory database that lives as long as this notebook

def q(sql):
    """Run SQL and show the result as a table (statements that return nothing show nothing)."""
    result = con.sql(sql)
    return None if result is None else result.df()

## 1. Loading and first look
`read_csv` reads a file straight into a query. It guesses the column types; you can override them. `CREATE TABLE ... AS SELECT` stores the result so later cells can use it. This is the SQL version of `pd.read_csv`.

In [ ]:
con.sql(f"""
CREATE OR REPLACE TABLE tx AS
SELECT * FROM read_csv('{find("card_transactions.csv")}', types = {{'ref_txn_id': 'VARCHAR'}})
""")
con.sql(f"CREATE OR REPLACE TABLE cu AS SELECT * FROM read_csv('{find('customer_master.csv')}')")
con.sql(f"CREATE OR REPLACE TABLE fx AS SELECT * FROM read_csv('{find('fx_rates.csv')}')")

q("""
SELECT 'tx' AS tbl, count(*) AS n_rows FROM tx
UNION ALL SELECT 'cu', count(*) FROM cu
UNION ALL SELECT 'fx', count(*) FROM fx
""")

A quick note on the notebook: each cell is `q(""" ...SQL... """)`. The SQL between the triple quotes is ordinary SQL; you could paste it into any DuckDB client.

In [ ]:
q(r"""
SELECT * FROM tx LIMIT 5
""")

In [ ]:
q(r"""
SELECT * FROM tx USING SAMPLE 5 ROWS    -- random rows (pandas: sample)
""")

In [ ]:
q(r"""
DESCRIBE tx      -- column names and types (pandas: dtypes)
""")

In [ ]:
q(r"""
SUMMARIZE tx     -- count, min, max, mean, quartiles, distinct values and nulls for every column
""")

In [ ]:
q(r"""
SELECT count(*)                       AS n_rows,
       count(DISTINCT txn_id)         AS distinct_ids,
       count(DISTINCT card_id)        AS distinct_cards,
       count(*) - count(ref_txn_id)   AS missing_ref_txn_id   -- count(col) ignores NULLs
FROM tx
""")

In [ ]:
q(r"""
SELECT status, count(*) AS n,
       round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct
FROM tx
GROUP BY status
ORDER BY n DESC
""")

In [ ]:
q(r"""
SELECT currency, count(*) AS n
FROM tx
GROUP BY currency
ORDER BY n DESC      -- note the dirty codes: 'gbp', 'GBP '
""")

In [ ]:
# Read only some columns and some rows while exploring a file (nothing is stored)
q(f"""
SELECT txn_id, amount, currency
FROM read_csv('{find("card_transactions.csv")}')
LIMIT 5
""")

**Use when:** the first five minutes with any table. Look for wrong types, unexpected categories, NULLs and odd ranges before you plan anything. `SUMMARIZE` does most of it in one line.

## 2. Selecting and filtering
`SELECT` chooses columns, `WHERE` chooses rows. Conditions combine with `AND`, `OR`, `NOT`. DuckDB adds handy extras: `* EXCLUDE`, `* REPLACE`, `* RENAME` and `COLUMNS()`.

In [ ]:
q(r"""
SELECT txn_id, amount FROM tx LIMIT 3     -- some columns
""")

In [ ]:
q(r"""
SELECT * FROM tx WHERE amount > 500 LIMIT 3
""")

In [ ]:
q(r"""
SELECT count(*) AS n FROM tx WHERE status = 'POSTED' AND amount > 500    -- AND, OR, NOT; use brackets to be safe
""")

In [ ]:
q(r"""
SELECT count(*) AS n FROM tx WHERE currency IN ('EUR', 'USD')            -- value in a list
""")

In [ ]:
q(r"""
SELECT count(*) AS n FROM tx WHERE amount BETWEEN 100 AND 200            -- inclusive at both ends
""")

In [ ]:
q(r"""
SELECT count(*) AS n FROM tx WHERE merchant LIKE 'TESCO%'                -- % = any characters; ILIKE ignores case
""")

In [ ]:
q(r"""
SELECT count(*) AS n FROM tx WHERE merchant ILIKE '%tesco%'
""")

In [ ]:
q(r"""
SELECT txn_id, amount, status FROM tx WHERE amount = 0 LIMIT 5          -- filter rows AND choose columns
""")

In [ ]:
q(r"""
SELECT * EXCLUDE (ref_txn_id, file_date) FROM tx LIMIT 2                 -- all columns except these (pandas: drop)
""")

In [ ]:
q(r"""
SELECT * RENAME (txn_timestamp AS ts_raw) FROM tx LIMIT 2                -- rename one column, keep the rest
""")

In [ ]:
q(r"""
SELECT * REPLACE (upper(trim(currency)) AS currency) FROM tx LIMIT 2     -- change a column in place, keep the rest
""")

In [ ]:
q(r"""
SELECT txn_id AS id, amount * 2 AS double_amount FROM tx LIMIT 3         -- AS gives a new name
""")

In [ ]:
q(r"""
SELECT COLUMNS('amount|currency') FROM tx LIMIT 3                        -- columns whose names match a pattern
""")

In [ ]:
q(r"""
SELECT DISTINCT currency FROM tx ORDER BY currency                       -- distinct values (pandas: unique)
""")

In [ ]:
q(r"""
SELECT txn_id FROM tx ORDER BY txn_id LIMIT 3 OFFSET 10                  -- skip 10 rows, take 3
""")

**Views and copies.** A query never changes the table it reads. To keep a result, store it: `CREATE TABLE name AS SELECT ...` (a stored copy) or `CREATE VIEW name AS SELECT ...` (a saved query that re-runs each time). Dbt models are views or tables built this way.

In [ ]:
q(r"""
CREATE OR REPLACE VIEW posted AS SELECT * FROM tx WHERE status = 'POSTED'
""")

In [ ]:
q(r"""
SELECT count(*) AS n FROM posted
""")

**Use when:** any time you need a subset. Filter early: `WHERE` before joins and aggregation makes everything after it cheaper.

## 3. Data types
Wrong types cause silent errors: numbers stored as text sort wrongly, dates stored as text cannot be subtracted. In SQL, `CAST(x AS type)` (or `x::type`) converts and fails if it cannot; `TRY_CAST` returns NULL instead.

In [ ]:
q(r"""
CREATE OR REPLACE TABLE messy AS
SELECT * FROM (VALUES
    ('1', '10.50', 'yes'),
    ('2', '£20.00', 'no'),
    ('3', 'n/a',   'yes'),
    ('4', '1,200', 'no')
) AS t(id, amount, flag)
""")

In [ ]:
q(r"""
SELECT *, typeof(id) AS id_type, typeof(amount) AS amount_type FROM messy
""")

In [ ]:
q(r"""
SELECT id::INTEGER AS id_int, CAST(id AS BIGINT) AS id_big FROM messy      -- :: is shorthand for CAST
""")

In [ ]:
q(r"""
SELECT amount,
       TRY_CAST(amount AS DOUBLE) AS try_cast      -- anything that cannot convert becomes NULL, not an error
FROM messy
""")

In [ ]:
q(r"""
-- Clean the text first, then convert
SELECT amount,
       TRY_CAST(replace(replace(amount, '£', ''), ',', '') AS DOUBLE) AS amount_num
FROM messy
""")

In [ ]:
q(r"""
SELECT flag, flag = 'yes' AS flag_bool, CASE flag WHEN 'yes' THEN TRUE ELSE FALSE END AS flag_case FROM messy
""")

**Money: use DECIMAL, not DOUBLE.** Floating point cannot store 0.1 exactly, so sums drift by tiny amounts. `DECIMAL(12,2)` stores pounds and pence exactly. (pandas does not have this choice built in; the card data arrives as DOUBLE, so we round when we report.)

In [ ]:
q(r"""
SELECT 0.1 + 0.2 AS double_sum, 0.1::DECIMAL(12,2) + 0.2::DECIMAL(12,2) AS decimal_sum
""")

In [ ]:
q(r"""
SELECT amount, amount::DECIMAL(12,2) AS amount_exact FROM tx LIMIT 3
""")

An `ENUM` restricts a column to a fixed set of values and stores them compactly (pandas: `category`).

In [ ]:
q(r"""
CREATE OR REPLACE TYPE status_t AS ENUM ('PENDING', 'POSTED', 'REVERSED')
""")

In [ ]:
q(r"""
SELECT status::status_t AS status FROM tx LIMIT 3
""")

In [ ]:
q(r"""
SELECT TRY_CAST('2026-13-45' AS DATE) AS bad_date, TRY_CAST('2026-03-31' AS DATE) AS good_date
""")

In [ ]:
q(r"""
-- Check what failed to convert: count the NULLs that TRY_CAST created
SELECT count(*) AS failed
FROM messy
WHERE amount IS NOT NULL AND TRY_CAST(amount AS DOUBLE) IS NULL
""")

In [ ]:
q(r"""
DESCRIBE SELECT txn_id, amount, file_date FROM tx   -- DuckDB read file_date as a real DATE already
""")

**Use when:** straight after loading. Check types with `DESCRIBE`, convert, and use `TRY_CAST` followed by a count of the new NULLs so you can see what failed.

## 4. Missing values
Missing data is `NULL`. A NULL means *unknown*, so SQL treats it specially: `NULL = NULL` is not true, any arithmetic with NULL gives NULL, and `WHERE x = NULL` returns nothing. Test with `IS NULL` / `IS NOT NULL`.

The card data has few real gaps, so this section uses a small table.

In [ ]:
q(r"""
CREATE OR REPLACE TABLE gaps AS
SELECT * FROM (VALUES
    ('C1', DATE '2026-03-01', 100.0, 'ok'),
    ('C1', DATE '2026-03-02', NULL,  NULL),
    ('C1', DATE '2026-03-03', 120.0, 'ok'),
    ('C2', DATE '2026-03-04', NULL,  ''),
    ('C2', DATE '2026-03-05', NULL,  'ok'),
    ('C3', DATE '2026-03-06', 50.0,  NULL)
) AS t(card_id, day, balance, note)
""")

In [ ]:
q(r"""
SELECT * FROM gaps
""")

In [ ]:
q(r"""
SELECT NULL = NULL AS equals, NULL IS NULL AS is_null, 5 + NULL AS arithmetic, NULL IS NOT DISTINCT FROM NULL AS null_safe_equals
""")

In [ ]:
q(r"""
SELECT * FROM gaps WHERE balance IS NULL           -- the rows with a missing balance
""")

In [ ]:
q(r"""
SELECT count(*) AS rows,
       count(balance) AS balance_present,                 -- count(col) skips NULLs
       count(*) - count(balance) AS balance_missing,
       round(100.0 * (count(*) - count(balance)) / count(*), 1) AS pct_missing
FROM gaps
""")

In [ ]:
q(r"""
SELECT * FROM gaps WHERE balance IS NOT NULL       -- drop rows with a missing value (pandas: dropna)
""")

In [ ]:
q(r"""
SELECT card_id, day, coalesce(balance, 0) AS balance FROM gaps            -- fill with a constant (pandas: fillna)
""")

In [ ]:
q(r"""
SELECT card_id, day, coalesce(balance, avg(balance) OVER ()) AS balance FROM gaps    -- fill with a statistic
""")

In [ ]:
q(r"""
-- Carry the last known value forward (the FX weekend rule): last_value ... IGNORE NULLS
-- (look at C2: it picks up C1's balance, because nothing says the groups are separate. The next query fixes that.)
SELECT card_id, day, balance,
       last_value(balance IGNORE NULLS) OVER (ORDER BY day) AS ffilled
FROM gaps
ORDER BY day
""")

In [ ]:
q(r"""
-- Forward fill WITHIN each group, so one card's balance never leaks into another's
SELECT card_id, day, balance,
       last_value(balance IGNORE NULLS) OVER (PARTITION BY card_id ORDER BY day) AS ffilled
FROM gaps
ORDER BY card_id, day
""")

In [ ]:
q(r"""
-- Carry the next known value backward
SELECT day, balance,
       first_value(balance IGNORE NULLS) OVER (ORDER BY day ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS bfilled
FROM gaps
ORDER BY day
""")

Empty strings are **not** NULL in SQL either. `NULLIF(x, '')` turns an empty string into NULL; `COALESCE` then fills the first non-NULL value.

In [ ]:
q(r"""
SELECT note,
       note IS NULL AS is_null,
       NULLIF(note, '') IS NULL AS is_null_after_nullif,
       coalesce(NULLIF(note, ''), card_id) AS filled
FROM gaps
""")

In [ ]:
q(r"""
-- Keep a record of what you filled
SELECT card_id, day,
       balance IS NULL AS balance_was_missing,
       last_value(balance IGNORE NULLS) OVER (PARTITION BY card_id ORDER BY day) AS balance
FROM gaps
ORDER BY card_id, day
""")

There is no built-in `interpolate` (straight-line fill) in SQL. Forward fill and a flag column cover most cases; linear interpolation needs both neighbours joined on and some arithmetic.

**Use when:** after loading and again after joins (a `LEFT JOIN` creates NULLs). Record what you filled so a user can tell real data from filled data.

## 5. Duplicates
Decide first what makes two rows "the same". `SELECT DISTINCT *` removes exact duplicates. Duplicates on a *key* (the same id with a different status) need a rule for which row to keep.

In [ ]:
q(r"""
SELECT (SELECT count(*) FROM tx) - (SELECT count(*) FROM (SELECT DISTINCT * FROM tx)) AS exact_duplicate_rows
""")

In [ ]:
q(r"""
-- See the duplicated rows: group by every column and keep groups bigger than 1
SELECT *, count(*) AS copies
FROM tx
GROUP BY ALL
HAVING count(*) > 1
ORDER BY txn_id
LIMIT 6
""")

In [ ]:
q(r"""
SELECT count(*) AS rows_after_distinct FROM (SELECT DISTINCT * FROM tx)
""")

In [ ]:
q(r"""
SELECT count(*) AS rows, count(DISTINCT txn_id) AS distinct_ids, count(*) - count(DISTINCT txn_id) AS repeated_ids
FROM (SELECT DISTINCT * FROM tx)
""")

In [ ]:
q(r"""
-- DISTINCT ON keeps ONE row per key: the first in the ORDER BY. Order so the latest comes first.
SELECT DISTINCT ON (txn_id) *
FROM tx
ORDER BY txn_id, file_date DESC
""")

In [ ]:
q(r"""
-- The same with QUALIFY, which filters on a window function (no subquery needed)
SELECT *
FROM (SELECT DISTINCT * FROM tx)
QUALIFY row_number() OVER (PARTITION BY txn_id ORDER BY file_date DESC) = 1
""")

In [ ]:
q(r"""
-- Same file date, two statuses? Add a tie-break to the ORDER BY.
CREATE OR REPLACE TABLE tx_latest AS
SELECT *
FROM (SELECT DISTINCT * FROM tx)
QUALIFY row_number() OVER (
    PARTITION BY txn_id
    ORDER BY file_date DESC,
             CASE status WHEN 'PENDING' THEN 0 WHEN 'POSTED' THEN 1 WHEN 'REVERSED' THEN 2 END DESC
) = 1
""")

In [ ]:
q(r"""
SELECT count(*) AS rows, count(DISTINCT txn_id) AS ids FROM tx_latest      -- the two should be equal
""")

In [ ]:
q(r"""
SELECT txn_id FROM tx_latest GROUP BY txn_id HAVING count(*) > 1           -- a uniqueness test: should return no rows
""")

In [ ]:
q(r"""
-- arg_max: the value of one column at the maximum of another (the status on the latest file date)
SELECT txn_id, arg_max(status, file_date) AS latest_status
FROM tx
GROUP BY txn_id
LIMIT 3
""")

**Use when:** the source can resend data. Count before and after, and check whether the rows you drop are really identical. The `HAVING count(*) > 1` query is exactly what dbt's `unique` test runs.

## 6. Text cleaning
String functions work on a whole column at once. Most take the string first. Regular expressions use `regexp_*` functions. There is no `initcap` in DuckDB, so title case is built from `upper`, `lower` and `substr`.

In [ ]:
q(r"""
SELECT DISTINCT currency FROM tx ORDER BY 1      -- spot the problems first
""")

In [ ]:
q(r"""
SELECT upper(trim(currency)) AS currency, count(*) AS n FROM tx GROUP BY ALL ORDER BY n DESC
""")

In [ ]:
q(r"""
SELECT segment,
       upper(substr(trim(segment), 1, 1)) || lower(substr(trim(segment), 2)) AS segment_clean
FROM (SELECT DISTINCT segment FROM cu)
""")

In [ ]:
q(r"""
SELECT lower('Premium') AS lower_, upper('Premium') AS upper_, length('Premium') AS len, 'Pre' || 'mium' AS joined
""")

In [ ]:
q(r"""
SELECT merchant, count(*) AS n FROM tx GROUP BY merchant ORDER BY n DESC LIMIT 8    -- note the store numbers
""")

In [ ]:
q(r"""
-- Collapse repeated spaces with a regular expression ('g' = replace every match)
SELECT regexp_replace('TESCO   STORES  4848', '\s+', ' ', 'g') AS tidy
""")

In [ ]:
q(r"""
-- Remove trailing store numbers: 'TESCO STORES 4848' -> 'TESCO STORES'
SELECT count(DISTINCT merchant) AS raw_variants,
       count(DISTINCT trim(regexp_replace(merchant, '\s*\d+$', ''))) AS cleaned_variants
FROM tx
""")

In [ ]:
q(r"""
SELECT merchant, regexp_extract(merchant, '(\d+)$', 1) AS store_number FROM tx LIMIT 5    -- capture group 1
""")

In [ ]:
q(r"""
SELECT count(*) FILTER (WHERE starts_with(merchant, 'TESCO')) AS starts_tesco,
       count(*) FILTER (WHERE contains(merchant, 'S/MKT'))   AS contains_smkt
FROM tx
""")

In [ ]:
q(r"""
SELECT txn_timestamp, split_part(txn_timestamp, ' ', 1) AS date_part, split_part(txn_timestamp, ' ', 2) AS time_part FROM tx LIMIT 3
""")

In [ ]:
q(r"""
SELECT (file_date::VARCHAR)[1:7] AS year_month FROM tx LIMIT 3       -- slice characters: [start:end], 1-based, both included
""")

In [ ]:
q(r"""
SELECT unnest(string_split('a,b', ',')) AS item                     -- one row per item (pandas: str.split + explode)
""")

In [ ]:
q(r"""
-- Standardise by mapping variants to one value: a small lookup table, then a join
CREATE OR REPLACE TABLE currency_map AS
SELECT * FROM (VALUES ('GBP','GBP'), ('EUR','EUR'), ('USD','USD'), ('CHF','CHF')) AS t(raw_clean, currency_std)
""")

In [ ]:
q(r"""
SELECT m.currency_std, count(*) AS n
FROM tx
LEFT JOIN currency_map AS m ON upper(trim(tx.currency)) = m.raw_clean
GROUP BY ALL
ORDER BY n DESC
""")

In [ ]:
q(r"""
-- Check nothing was lost: unmapped values come out as NULL, so count them
SELECT count(*) AS unmapped
FROM tx LEFT JOIN currency_map AS m ON upper(trim(tx.currency)) = m.raw_clean
WHERE m.currency_std IS NULL
""")

**Use when:** labels typed by humans or different systems. Clean into a new column when the original might still be needed (keep `merchant`, add `merchant_clean`).

## 7. Recoding and creating columns
New columns are computed in the `SELECT` list. `CASE WHEN` is the SQL if/else (pandas: `np.where`, `np.select`). DuckDB lets a later column in the same `SELECT` use an earlier alias, which keeps long expressions readable.

In [ ]:
q(r"""
SELECT txn_id, status,
       status = 'REVERSED' AS is_reversal,          -- a True/False flag
       amount = 0          AS is_zero,
       amount
FROM tx LIMIT 3
""")

In [ ]:
q(r"""
SELECT CASE WHEN amount < 0 THEN 'refund' ELSE 'purchase' END AS direction, count(*) AS n
FROM tx GROUP BY ALL
""")

In [ ]:
q(r"""
-- CASE with several conditions: the first match wins
SELECT CASE WHEN amount < 0    THEN 'refund'
            WHEN amount = 0    THEN 'zero'
            WHEN amount < 50   THEN 'small'
            WHEN amount < 200  THEN 'medium'
            ELSE 'large' END AS size,
       count(*) AS n
FROM tx GROUP BY ALL ORDER BY n DESC
""")

In [ ]:
q(r"""
-- Equal-sized bands: ntile(4) splits the rows into four groups of (nearly) equal count (pandas: qcut)
SELECT quartile, min(amount) AS from_amount, max(amount) AS to_amount, count(*) AS n
FROM (SELECT amount, ntile(4) OVER (ORDER BY amount) AS quartile FROM tx)
GROUP BY quartile ORDER BY quartile
""")

In [ ]:
q(r"""
-- Recode with a CASE expression on a single column (pandas: map)
SELECT status,
       CASE status WHEN 'PENDING' THEN 0 WHEN 'POSTED' THEN 1 WHEN 'REVERSED' THEN 2 END AS status_rank,
       count(*) AS n
FROM tx GROUP BY ALL ORDER BY status_rank
""")

In [ ]:
q(r"""
SELECT min(amount) AS raw_min, min(greatest(amount, 0)) AS floored_at_zero, max(least(amount, 500)) AS capped_at_500 FROM tx   -- clip
""")

In [ ]:
q(r"""
SELECT amount, round(amount, 1) AS rounded, abs(amount) AS absolute, amount * 0.8 AS scaled FROM tx LIMIT 3
""")

In [ ]:
q(r"""
-- Reuse an alias in the same SELECT
SELECT amount,
       amount * 0.8        AS gbp_estimate,
       gbp_estimate * 1.2  AS with_margin
FROM tx LIMIT 3
""")

In [ ]:
q(r"""
SELECT if(amount > 100, 'big', 'normal') AS size, count(*) AS n FROM tx GROUP BY ALL    -- short form for a two-way CASE
""")

To change values in a stored table, use `UPDATE`. It changes the table in place, so try it on a copy.

In [ ]:
q(r"""
CREATE OR REPLACE TABLE tx_copy AS SELECT * FROM tx
""")

In [ ]:
q(r"""
UPDATE tx_copy SET status = 'ZERO' WHERE amount = 0
""")

In [ ]:
q(r"""
SELECT status, count(*) AS n FROM tx_copy GROUP BY status ORDER BY n DESC
""")

In [ ]:
q(r"""
DROP TABLE tx_copy
""")

**Use when:** every transformation step. Write the rule in plain words first (for example *a reversal is a row whose status is REVERSED*), then translate it into one of these. In dbt you would not `UPDATE`: you write a new SELECT with the change in it.

## 8. Dates and times
Dates arrive as text in many formats. Convert once, early, to proper `DATE` / `TIMESTAMP` values; after that you can extract parts, subtract, group by month and join on date.

In [ ]:
q(r"""
SELECT txn_timestamp FROM tx USING SAMPLE 6 ROWS        -- two formats are mixed
""")

In [ ]:
q(r"""
-- try_strptime returns NULL where the format does not fit; coalesce takes the first that worked
SELECT count(*) AS rows,
       count(try_strptime(txn_timestamp, '%Y-%m-%d %H:%M:%S')) AS iso_format,
       count(try_strptime(txn_timestamp, '%d/%m/%Y %H:%M'))     AS day_first,
       count(*) - count(coalesce(try_strptime(txn_timestamp, '%Y-%m-%d %H:%M:%S'),
                                 try_strptime(txn_timestamp, '%d/%m/%Y %H:%M'))) AS could_not_parse
FROM tx
""")

In [ ]:
q(r"""
CREATE OR REPLACE VIEW tx3 AS
SELECT *,
       coalesce(try_strptime(txn_timestamp, '%Y-%m-%d %H:%M:%S'),
                try_strptime(txn_timestamp, '%d/%m/%Y %H:%M')) AS ts
FROM tx
""")

In [ ]:
q(r"""
DESCRIBE SELECT ts FROM tx3
""")

In [ ]:
q(r"""
SELECT ts,
       year(ts) AS year, month(ts) AS month, day(ts) AS day, hour(ts) AS hour,
       dayname(ts) AS weekday, isodow(ts) >= 6 AS is_weekend
FROM tx3 LIMIT 4
""")

In [ ]:
q(r"""
SELECT ts, ts::DATE AS date_only, date_trunc('month', ts)::DATE AS month_start, strftime(ts, '%Y-%m') AS year_month FROM tx3 LIMIT 3
""")

In [ ]:
q(r"""
-- Group by month: the month the purchase was MADE
SELECT date_trunc('month', ts)::DATE AS month, count(*) AS n, round(sum(amount), 0) AS total
FROM tx3 GROUP BY ALL ORDER BY month
""")

In [ ]:
q(r"""
-- Date arithmetic: how many days after the purchase did the row arrive in a file?
SELECT date_diff('day', ts::DATE, file_date) AS days_late, count(*) AS n
FROM tx3 GROUP BY ALL ORDER BY days_late LIMIT 8
""")

In [ ]:
q(r"""
SELECT count(*) AS n FROM tx3 WHERE ts >= DATE '2026-03-01' AND ts < DATE '2026-04-01'    -- filter by date: >= start, < next start
""")

In [ ]:
q(r"""
SELECT ts, ts + INTERVAL 1 DAY AS plus_a_day, ts + INTERVAL 1 MONTH AS plus_a_month FROM tx3 LIMIT 2
""")

**A full calendar.** `generate_series` makes one row per day. Join the data onto it and the gaps appear. This is how the weekend FX gaps are found.

In [ ]:
q(r"""
WITH cal AS (
    SELECT d::DATE AS rate_date
    FROM generate_series((SELECT min(rate_date) FROM fx), DATE '2026-03-31', INTERVAL 1 DAY) AS t(d)
)
SELECT count(*) AS calendar_days,
       count(*) FILTER (WHERE rate_date IN (SELECT rate_date FROM fx)) AS days_with_rates,
       count(*) FILTER (WHERE rate_date NOT IN (SELECT rate_date FROM fx)) AS days_without
FROM cal
""")

In [ ]:
q(r"""
-- Fill the gaps: one row per day and currency, carrying the last published rate forward
CREATE OR REPLACE TABLE fx_daily AS
WITH cal AS (
    SELECT d::DATE AS rate_date
    FROM generate_series((SELECT min(rate_date) FROM fx), DATE '2026-03-31', INTERVAL 1 DAY) AS t(d)
),
grid AS (SELECT cal.rate_date, c.currency FROM cal CROSS JOIN (SELECT DISTINCT currency FROM fx) AS c)
SELECT g.rate_date, g.currency,
       f.rate_to_gbp IS NULL AS rate_is_filled,
       last_value(f.rate_to_gbp IGNORE NULLS) OVER (PARTITION BY g.currency ORDER BY g.rate_date) AS rate_to_gbp
FROM grid AS g
LEFT JOIN fx AS f ON f.rate_date = g.rate_date AND f.currency = g.currency
""")

In [ ]:
q(r"""
SELECT currency, count(*) AS days, count(*) FILTER (WHERE rate_is_filled) AS filled_days, count(*) FILTER (WHERE rate_to_gbp IS NULL) AS still_missing FROM fx_daily GROUP BY currency
""")

In [ ]:
q(r"""
SELECT time_bucket(INTERVAL 7 DAY, ts::DATE) AS week_start, count(*) AS n FROM tx3 GROUP BY ALL ORDER BY week_start LIMIT 4    -- group into weeks
""")

Time zones need the ICU extension (`INSTALL icu; LOAD icu;`) and `AT TIME ZONE`. The card data has no time zone, so it is not shown here.

**Use when:** any date column. Decide which date you mean (purchase date, file date, posting date) before you group by month, because the choice changes the totals.

## 9. Sorting and ranking
`ORDER BY` sorts the result (it is the *last* thing a query does, so it cannot be relied on inside a subquery). Ranking uses window functions: `row_number`, `rank`, `dense_rank`.

In [ ]:
q(r"""
SELECT txn_id, amount FROM tx ORDER BY amount DESC LIMIT 3
""")

In [ ]:
q(r"""
SELECT status, amount, txn_id FROM tx ORDER BY status ASC, amount DESC LIMIT 3         -- several keys
""")

In [ ]:
q(r"""
SELECT card_id, ref_txn_id FROM tx ORDER BY ref_txn_id DESC NULLS LAST LIMIT 3        -- say where NULLs go
""")

In [ ]:
q(r"""
-- Top and bottom 3 (pandas: nlargest / nsmallest)
(SELECT txn_id, amount, 'top' AS which FROM tx ORDER BY amount DESC LIMIT 3)
UNION ALL
(SELECT txn_id, amount, 'bottom' FROM tx ORDER BY amount LIMIT 3)
""")

In [ ]:
q(r"""
-- row_number: every row a different rank; rank: ties share a rank and leave gaps; dense_rank: ties share, no gaps
SELECT amount,
       row_number() OVER (ORDER BY amount DESC) AS row_number,
       rank()       OVER (ORDER BY amount DESC) AS rank,
       dense_rank() OVER (ORDER BY amount DESC) AS dense_rank
FROM (SELECT * FROM tx WHERE amount IN (SELECT amount FROM tx GROUP BY amount HAVING count(*) > 1 ORDER BY amount DESC LIMIT 3))
ORDER BY amount DESC LIMIT 8
""")

In [ ]:
q(r"""
-- Rank within a group: the biggest purchase per card
SELECT card_id, txn_id, amount
FROM tx
QUALIFY row_number() OVER (PARTITION BY card_id ORDER BY amount DESC) = 1
ORDER BY card_id LIMIT 4
""")

In [ ]:
q(r"""
-- Top 2 per card: QUALIFY with <= 2
SELECT card_id, txn_id, amount
FROM tx
QUALIFY row_number() OVER (PARTITION BY card_id ORDER BY amount DESC) <= 2
ORDER BY card_id, amount DESC LIMIT 6
""")

In [ ]:
q(r"""
-- Tie-break by a custom order (the same trick the silver pipeline uses for status)
SELECT count(*) AS n_after_dedup
FROM (
    SELECT * FROM (SELECT DISTINCT * FROM tx)
    QUALIFY row_number() OVER (
        PARTITION BY txn_id
        ORDER BY file_date DESC, CASE status WHEN 'PENDING' THEN 0 WHEN 'POSTED' THEN 1 ELSE 2 END DESC) = 1
)
""")

**Use when:** choosing one row per group. Rank with an explicit `ORDER BY` and filter on `= 1`; never rely on the order the file arrived in.

## 10. Grouping and aggregating
`GROUP BY` collapses rows into groups and aggregate functions summarise each group. Every selected column must be either in the `GROUP BY` or inside an aggregate (`GROUP BY ALL` does the first part for you). `WHERE` filters rows *before* grouping; `HAVING` filters groups *after*.

In [ ]:
q(r"""
SELECT status, sum(amount) AS total FROM tx GROUP BY status
""")

In [ ]:
q(r"""
SELECT status, count(*) AS n, round(sum(amount), 0) AS total, round(avg(amount), 2) AS mean, min(amount) AS min, max(amount) AS max
FROM tx GROUP BY status
""")

In [ ]:
q(r"""
SELECT status,
       count(*)                AS n,
       sum(amount)             AS total,
       count(DISTINCT card_id) AS cards
FROM tx GROUP BY status
""")

In [ ]:
q(r"""
SELECT currency, status, count(*) AS n FROM tx GROUP BY ALL ORDER BY n DESC LIMIT 6          -- several keys
""")

In [ ]:
q(r"""
-- Conditional aggregates with FILTER (pandas: boolean mask before the aggregate)
SELECT count(*) AS n,
       count(*) FILTER (WHERE status = 'POSTED')   AS posted,
       count(*) FILTER (WHERE status = 'REVERSED') AS reversed,
       sum(amount) FILTER (WHERE amount > 100)      AS total_over_100
FROM tx
""")

In [ ]:
q(r"""
SELECT card_id, count(*) AS n FROM tx GROUP BY card_id HAVING count(*) > 100 ORDER BY n DESC LIMIT 5   -- filter on the group result
""")

In [ ]:
q(r"""
-- A window function keeps the row level but adds the group figure (pandas: transform)
SELECT card_id, amount,
       sum(amount) OVER (PARTITION BY card_id) AS card_total,
       round(amount / sum(amount) OVER (PARTITION BY card_id), 4) AS share_of_card
FROM tx LIMIT 3
""")

In [ ]:
q(r"""
-- Pivot-style summary done by hand with FILTER
SELECT status,
       round(sum(amount) FILTER (WHERE upper(trim(currency)) = 'GBP'), 0) AS gbp,
       round(sum(amount) FILTER (WHERE upper(trim(currency)) = 'EUR'), 0) AS eur,
       round(sum(amount) FILTER (WHERE upper(trim(currency)) = 'USD'), 0) AS usd
FROM tx GROUP BY status
""")

In [ ]:
q(r"""
SELECT upper(trim(currency)) AS currency, count(*) AS n,
       median(amount) AS median, quantile_cont(amount, 0.9) AS p90
FROM tx GROUP BY ALL ORDER BY n DESC
""")

In [ ]:
q(r"""
-- Totals and subtotals in one query
SELECT status, upper(trim(currency)) AS currency, count(*) AS n
FROM tx
GROUP BY ROLLUP (status, upper(trim(currency)))
ORDER BY status NULLS LAST, currency NULLS LAST LIMIT 8
""")

In [ ]:
q(r"""
-- Collect values into a list, or into text
SELECT status, list(DISTINCT upper(trim(currency)) ORDER BY upper(trim(currency))) AS currencies,
       string_agg(DISTINCT upper(trim(currency)), ', ') AS as_text
FROM tx GROUP BY status
""")

In [ ]:
q(r"""
-- first / last in each group by a chosen order
SELECT txn_id,
       first(status ORDER BY file_date) AS first_status,
       last(status ORDER BY file_date)  AS last_status
FROM tx GROUP BY txn_id LIMIT 3
""")

**Use when:** building summaries. Check that the rows in equal the rows accounted for in the groups (the totals must reconcile). A gold table is a `GROUP BY` over a silver table.

## 11. Window and running calculations
Window functions look across neighbouring rows without collapsing them: previous value, running total, moving average. The `OVER (...)` clause says which rows, in what order. The frame (`ROWS BETWEEN ...`) says how many neighbours.

In [ ]:
q(r"""
CREATE OR REPLACE TABLE daily AS
SELECT ts::DATE AS day, sum(amount) AS amount
FROM tx3 WHERE status = 'POSTED'
GROUP BY ALL
""")

In [ ]:
q(r"""
SELECT * FROM daily ORDER BY day LIMIT 3
""")

In [ ]:
q(r"""
SELECT day, amount, sum(amount) OVER (ORDER BY day) AS running_total FROM daily ORDER BY day LIMIT 3          -- cumulative sum
""")

In [ ]:
q(r"""
SELECT day, amount,
       lag(amount) OVER (ORDER BY day)               AS previous,
       amount - lag(amount) OVER (ORDER BY day)      AS change,
       lead(amount) OVER (ORDER BY day)              AS next_day
FROM daily ORDER BY day LIMIT 3
""")

In [ ]:
q(r"""
SELECT day, amount,
       round(100 * (amount / lag(amount) OVER (ORDER BY day) - 1), 1) AS pct_change
FROM daily ORDER BY day LIMIT 3
""")

In [ ]:
q(r"""
-- 7-day moving average: the current row and the 6 before it
SELECT day, amount,
       round(avg(amount) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS avg_7d,
       count(*) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS rows_in_window
FROM daily ORDER BY day LIMIT 8
""")

In [ ]:
q(r"""
SELECT day, amount, max(amount) OVER (ORDER BY day ROWS UNBOUNDED PRECEDING) AS max_so_far FROM daily ORDER BY day LIMIT 3      -- expanding
""")

In [ ]:
q(r"""
-- Within groups: the previous transaction amount for the same card
SELECT card_id, ts, amount,
       lag(amount) OVER (PARTITION BY card_id ORDER BY ts) AS prev_amount
FROM tx3
ORDER BY card_id, ts LIMIT 4
""")

In [ ]:
q(r"""
-- Time between consecutive transactions on a card
SELECT card_id, ts,
       ts - lag(ts) OVER (PARTITION BY card_id ORDER BY ts) AS gap
FROM tx3
ORDER BY card_id, ts LIMIT 4
""")

In [ ]:
q(r"""
-- Running count within a group
SELECT card_id, ts, row_number() OVER (PARTITION BY card_id ORDER BY ts) AS n_so_far
FROM tx3
ORDER BY card_id, ts LIMIT 4
""")

**Use when:** time-based features, finding gaps and detecting changes (for example a segment that changed between snapshots). Sort order inside `OVER` is what defines "previous".

## 12. Reshaping
`PIVOT` turns values into columns (long to wide); `UNPIVOT` turns columns into values (wide to long). `UNNEST` turns a list into rows.

In [ ]:
q(r"""
-- Long to wide: one column per currency
PIVOT fx ON currency USING first(rate_to_gbp) GROUP BY rate_date ORDER BY rate_date LIMIT 3
""")

In [ ]:
q(r"""
CREATE OR REPLACE TABLE fx_wide AS
PIVOT fx ON currency USING first(rate_to_gbp) GROUP BY rate_date
""")

In [ ]:
q(r"""
-- Wide to long: columns become rows
UNPIVOT fx_wide ON EUR, USD, CHF INTO NAME currency VALUE rate
ORDER BY rate_date, currency LIMIT 4
""")

In [ ]:
q(r"""
-- PIVOT with an aggregate over a summary (status down, currency across)
PIVOT (SELECT status, upper(trim(currency)) AS currency, amount FROM tx)
ON currency IN ('GBP', 'EUR', 'USD', 'CHF')
USING round(sum(amount), 0)
GROUP BY status
""")

In [ ]:
q(r"""
-- UNNEST: a list in one cell becomes one row per item
SELECT card, merchant
FROM (VALUES ('C1', ['TESCO', 'TFL']), ('C2', ['NETFLIX'])) AS t(card, merchants), unnest(merchants) AS u(merchant)
""")

In [ ]:
q(r"""
-- One-hot encoding by hand
SELECT txn_id, status = 'PENDING' AS s_pending, status = 'POSTED' AS s_posted, status = 'REVERSED' AS s_reversed
FROM tx LIMIT 4
""")

In [ ]:
q(r"""
-- Wide-to-long on columns named like sales_2025, sales_2026
UNPIVOT (SELECT 1 AS id, 10 AS sales_2025, 11 AS sales_2026)
ON COLUMNS(* EXCLUDE id) INTO NAME year VALUE sales
""")

**Use when:** a source is wide but you need long (or the reverse), for example a column per currency that must be joined on currency. In many warehouses `PIVOT` needs the column values written out; DuckDB can discover them.

## 13. Combining tables
`JOIN` matches rows on keys. The danger is not the syntax but joins that silently change the row count: a `LEFT JOIN` onto a table where the key is not unique multiplies rows. Always check the key and the row counts.

In [ ]:
q(r"""
CREATE OR REPLACE TABLE cards AS
SELECT card_id,
       upper(substr(trim(segment), 1, 1)) || lower(substr(trim(segment), 2)) AS segment,
       account_status
FROM cu
QUALIFY row_number() OVER (PARTITION BY card_id ORDER BY snapshot_date DESC) = 1
""")

In [ ]:
q(r"""
SELECT (SELECT count(*) FROM tx_latest) AS tx_rows,
       (SELECT count(*) FROM cards) AS card_rows,
       (SELECT count(*) - count(DISTINCT card_id) FROM cards) AS duplicate_card_keys
""")

In [ ]:
q(r"""
-- LEFT JOIN: keep every transaction, add customer columns where found
SELECT count(*) AS rows_after_join,
       count(*) FILTER (WHERE c.card_id IS NULL) AS no_customer_match
FROM tx_latest AS t LEFT JOIN cards AS c ON c.card_id = t.card_id
""")

In [ ]:
q(r"""
-- Anti-join: transactions whose card is NOT in the customer table
SELECT count(*) AS transactions, count(DISTINCT t.card_id) AS cards
FROM tx_latest AS t
WHERE NOT EXISTS (SELECT 1 FROM cards AS c WHERE c.card_id = t.card_id)
""")

In [ ]:
q(r"""
-- Semi-join: transactions whose card IS in the table, without adding columns (EXISTS never multiplies rows)
SELECT count(*) AS transactions
FROM tx_latest AS t
WHERE EXISTS (SELECT 1 FROM cards AS c WHERE c.card_id = t.card_id)
""")

In [ ]:
q(r"""
-- Join types: INNER (matches only), LEFT, FULL (everything)
SELECT 'inner' AS how, count(*) AS n FROM tx_latest t INNER JOIN cards c USING (card_id)
UNION ALL SELECT 'left', count(*) FROM tx_latest t LEFT JOIN cards c USING (card_id)
UNION ALL SELECT 'full', count(*) FROM tx_latest t FULL JOIN cards c USING (card_id)
""")

In [ ]:
q(r"""
-- Different key names, and columns with the same name from both sides
SELECT a.k, a.v AS v_a, b.v AS v_b
FROM (VALUES (1, 10), (2, 20)) AS a(k, v)
FULL JOIN (VALUES (1, 100), (3, 300)) AS b(key, v) ON a.k = b.key
""")

In [ ]:
q(r"""
-- Join on several keys, with the GBP rate added as 1.0: the FX rate for the transaction DATE and CURRENCY
WITH t AS (
    SELECT txn_id, amount, upper(trim(currency)) AS currency,
           coalesce(try_strptime(txn_timestamp, '%Y-%m-%d %H:%M:%S'),
                    try_strptime(txn_timestamp, '%d/%m/%Y %H:%M'))::DATE AS tdate
    FROM tx_latest
)
SELECT count(*) AS rows, count(*) FILTER (WHERE coalesce(f.rate_to_gbp, CASE WHEN t.currency = 'GBP' THEN 1.0 END) IS NULL) AS missing_rate
FROM t LEFT JOIN fx_daily AS f ON f.rate_date = t.tdate AND f.currency = t.currency
""")

In [ ]:
q(r"""
-- UNION ALL stacks tables with the same columns (pandas: concat); UNION also removes duplicates
SELECT txn_id, amount FROM (SELECT * FROM tx_latest ORDER BY txn_id LIMIT 2)
UNION ALL
SELECT txn_id, amount FROM (SELECT * FROM tx_latest ORDER BY txn_id DESC LIMIT 2)
""")

In [ ]:
q(r"""
-- EXCEPT / INTERSECT compare two sets: cards in transactions but not in customers
SELECT card_id FROM tx_latest
EXCEPT
SELECT card_id FROM cards
ORDER BY card_id LIMIT 5
""")

In [ ]:
q(r"""
-- ASOF JOIN: match each row to the most recent earlier row (the segment AT the time of the purchase)
SELECT t.card_id, t.file_date, s.snapshot_date, s.segment
FROM tx_latest AS t
ASOF LEFT JOIN (SELECT card_id, snapshot_date, upper(substr(trim(segment), 1, 1)) || lower(substr(trim(segment), 2)) AS segment FROM cu) AS s
  ON t.card_id = s.card_id AND t.file_date >= s.snapshot_date
ORDER BY t.card_id, t.file_date LIMIT 3
""")

In [ ]:
q(r"""
SELECT coalesce(NULL, NULL, 3) AS first_non_null      -- fill a gap from another column or table
""")

**Use when:** enriching one table with another. Check key uniqueness and compare row counts before and after every join. Most wrong totals in real pipelines come from a join that multiplied rows. In dbt this is a `relationships` test plus a `unique` test.

## 14. Validation and outliers
Cleaning is not finished until you have checked it. In SQL a check is a query that **returns the bad rows**: if it returns nothing, the check passes. This is exactly how dbt tests work.

In [ ]:
q(r"""
SELECT min(amount), quantile_cont(amount, 0.01) AS p01, quantile_cont(amount, 0.05) AS p05,
       median(amount) AS median, quantile_cont(amount, 0.95) AS p95, quantile_cont(amount, 0.99) AS p99, max(amount)
FROM tx
""")

In [ ]:
q(r"""
-- IQR rule: flag values far outside the middle 50%
WITH b AS (
    SELECT quantile_cont(amount, 0.25) AS q1, quantile_cont(amount, 0.75) AS q3 FROM tx
),
lim AS (SELECT q1 - 1.5 * (q3 - q1) AS low, q3 + 1.5 * (q3 - q1) AS high FROM b)
SELECT round(low, 1) AS low, round(high, 1) AS high,
       count(*) FILTER (WHERE amount < low OR amount > high) AS flagged,
       round(100.0 * count(*) FILTER (WHERE amount < low OR amount > high) / count(*), 1) AS pct
FROM tx, lim GROUP BY low, high
""")

In [ ]:
q(r"""
-- z-score: distance from the mean in standard deviations
SELECT count(*) FILTER (WHERE abs(z) > 4) AS beyond_4_sd
FROM (SELECT (amount - avg(amount) OVER ()) / stddev_samp(amount) OVER () AS z FROM tx)
""")

In [ ]:
q(r"""
-- Outliers are questions, not errors. Look before you remove.
SELECT txn_id, merchant, amount, status FROM tx ORDER BY amount DESC LIMIT 3
""")

A check that returns **no rows** passes. Run each one and look:

In [ ]:
q(r"""
SELECT txn_id FROM tx_latest GROUP BY txn_id HAVING count(*) > 1             -- unique txn_id
""")

In [ ]:
q(r"""
SELECT * FROM tx_latest WHERE card_id IS NULL OR amount IS NULL              -- no missing keys or amounts
""")

In [ ]:
q(r"""
SELECT * FROM tx_latest WHERE status NOT IN ('PENDING', 'POSTED', 'REVERSED') -- only known statuses
""")

In [ ]:
q(r"""
SELECT DISTINCT currency FROM tx_latest WHERE currency NOT IN ('GBP', 'EUR', 'USD', 'CHF')   -- returns rows on purpose: the codes are not cleaned yet
""")

The last check returns rows on purpose (`gbp`, `GBP `, ...). That is what a check is for. Cleaning fixes it:

In [ ]:
q(r"""
SELECT DISTINCT upper(trim(currency)) AS currency FROM tx_latest WHERE upper(trim(currency)) NOT IN ('GBP', 'EUR', 'USD', 'CHF')
""")

In [ ]:
q(r"""
-- All the checks in one table: PASS when no bad rows
SELECT 'one row per txn_id' AS check_name,
       CASE WHEN count(*) = 0 THEN 'PASS' ELSE 'FAIL' END AS result, count(*) AS bad_rows
FROM (SELECT txn_id FROM tx_latest GROUP BY txn_id HAVING count(*) > 1)
UNION ALL
SELECT 'no missing card_id', CASE WHEN count(*) = 0 THEN 'PASS' ELSE 'FAIL' END, count(*)
FROM tx_latest WHERE card_id IS NULL
UNION ALL
SELECT 'purchases are not negative', CASE WHEN count(*) = 0 THEN 'PASS' ELSE 'FAIL' END, count(*)
FROM tx_latest WHERE amount < 0 AND status <> 'REVERSED'
UNION ALL
SELECT 'every card is in the customer table', CASE WHEN count(*) = 0 THEN 'PASS' ELSE 'FAIL' END, count(*)
FROM tx_latest WHERE card_id NOT IN (SELECT card_id FROM cards)
""")

The last check fails, and this time it is a real finding, not a bug: 150 transactions are on cards that are missing from the customer table (8 cards). The pipeline keeps them with segment 'Unknown' rather than dropping them. In dbt this is the `relationships` test set to `warn`.

In [ ]:
q(r"""
-- Reconcile: rows in = rows out + rows removed on purpose
SELECT (SELECT count(*) FROM tx) AS rows_in,
       (SELECT count(*) FROM tx_latest) AS rows_out,
       (SELECT count(*) FROM tx) - (SELECT count(*) FROM (SELECT DISTINCT * FROM tx)) AS exact_duplicates,
       (SELECT count(*) FROM (SELECT DISTINCT * FROM tx)) - (SELECT count(*) FROM tx_latest) AS superseded_by_later_status
""")

In [ ]:
q(r"""
-- Cross-check a total two ways
SELECT round((SELECT sum(amount) FROM tx_latest WHERE status = 'POSTED'), 2) AS filter_version,
       round((SELECT sum(amount) FILTER (WHERE status = 'POSTED') FROM tx_latest), 2) AS filter_clause_version
""")

In Python you can turn a check into a hard stop: `assert con.execute("SELECT count(*) FROM bad_rows_query").fetchall()[0][0] == 0`.

In [ ]:
bad = con.execute("SELECT count(*) FROM (SELECT txn_id FROM tx_latest GROUP BY txn_id HAVING count(*) > 1)").fetchall()[0][0]
assert bad == 0, f"{bad} duplicated txn_id values"
print("check passed")

**Use when:** the last step of every cleaning stage, and again after every join. A check that never fails is still worth keeping; the one that fails saves you from a wrong number in a report.

## 15. Chaining steps into a pipeline
Long queries are written as a chain of `WITH` blocks (CTEs): each block is one named step that reads the previous one. This is the SQL version of `.pipe()` and of the `utilities.py` steps from the workshop. In dbt, each block that matters becomes its own model, and `ref()` connects them.

In [ ]:
q(r"""
CREATE OR REPLACE TABLE silver_transactions AS
WITH deduped AS (                          -- step 1: drop exact duplicate rows
    SELECT DISTINCT * FROM tx
),
cleaned AS (                               -- step 2: tidy text, read both timestamp formats
    SELECT txn_id, ref_txn_id, card_id,
           upper(regexp_replace(trim(merchant), '\s+', ' ', 'g')) AS merchant_clean,
           amount,
           upper(trim(currency)) AS currency,
           status,
           coalesce(try_strptime(txn_timestamp, '%Y-%m-%d %H:%M:%S'),
                    try_strptime(txn_timestamp, '%d/%m/%Y %H:%M')) AS ts,
           file_date
    FROM deduped
),
latest AS (                                -- step 3: keep the latest status per transaction
    SELECT * FROM cleaned
    QUALIFY row_number() OVER (
        PARTITION BY txn_id
        ORDER BY file_date DESC, CASE status WHEN 'PENDING' THEN 0 WHEN 'POSTED' THEN 1 ELSE 2 END DESC) = 1
),
fx_all AS (                                -- step 4: exchange rates, with GBP = 1
    SELECT rate_date, currency, rate_to_gbp FROM fx_daily
    UNION ALL
    SELECT DISTINCT rate_date, 'GBP', 1.0 FROM fx_daily
)
SELECT l.txn_id, l.ref_txn_id, l.card_id, l.merchant_clean, l.amount, l.currency, l.status, l.ts,
       strftime(l.ts, '%Y-%m')               AS month,
       coalesce(c.segment, 'Unknown')        AS segment,          -- step 5: join the segment
       c.card_id IS NOT NULL                 AS customer_known,
       f.rate_to_gbp                         AS rate,             -- step 6: join the rate, convert to pounds
       l.amount * f.rate_to_gbp              AS amount_gbp,
       l.status = 'REVERSED'                 AS is_reversal,
       l.amount = 0                          AS zero_amount
FROM latest AS l
LEFT JOIN cards  AS c ON c.card_id = l.card_id
LEFT JOIN fx_all AS f ON f.rate_date = l.ts::DATE AND f.currency = l.currency
""")

In [ ]:
q(r"""
SELECT count(*) AS rows, count(DISTINCT txn_id) AS unique_ids,
       count(*) FILTER (WHERE amount_gbp IS NULL) AS missing_gbp,
       count(*) FILTER (WHERE NOT customer_known) AS unknown_cards
FROM silver_transactions
""")

The gold layer is then a `GROUP BY` over silver:

In [ ]:
q(r"""
CREATE OR REPLACE VIEW gold_spend_by_month_segment AS
SELECT month, segment, round(sum(amount_gbp), 2) AS spend_gbp, count(*) AS transactions
FROM silver_transactions
WHERE status = 'POSTED'
GROUP BY month, segment
ORDER BY month, segment
""")

In [ ]:
q(r"""
SELECT * FROM gold_spend_by_month_segment
""")

In [ ]:
q(r"""
-- Log the row count at each step so you can see where rows go
SELECT 'raw' AS step, count(*) AS n FROM tx
UNION ALL SELECT 'after DISTINCT', count(*) FROM (SELECT DISTINCT * FROM tx)
UNION ALL SELECT 'after latest status', count(*) FROM tx_latest
UNION ALL SELECT 'posted only', count(*) FROM silver_transactions WHERE status = 'POSTED'
""")

**Views as layers.** `CREATE VIEW` saves a query under a name so the next query can read it. A bronze/silver/gold pipeline is a stack of views (or tables). dbt writes the `CREATE` for you, works out the order from the `ref()` calls, and runs the tests.

**Use when:** a query gets longer than about ten lines. Named steps are easier to test, reorder and explain.

## 16. Saving results, and moving between pandas and SQL
`COPY ... TO` writes a query result to a file. DuckDB can also query a pandas DataFrame directly, by name, and hand results back as a DataFrame, so you can mix the two tools in one notebook.

In [ ]:
q(r"""
COPY (SELECT * FROM silver_transactions) TO 'silver_transactions.csv' (HEADER, DELIMITER ',')
""")

In [ ]:
q(r"""
SELECT count(*) AS rows FROM read_csv('silver_transactions.csv')
""")

In [ ]:
q(r"""
COPY (SELECT * FROM silver_transactions) TO 'silver_transactions.parquet' (FORMAT parquet)       -- keeps types; smaller and faster than CSV
""")

In [ ]:
q(r"""
DESCRIBE SELECT ts, amount, is_reversal FROM 'silver_transactions.parquet'   -- types survive the round trip (a CSV would lose them)
""")

In [ ]:
# Query a file directly, without loading it first
import os
print("parquet KB:", os.path.getsize("silver_transactions.parquet") // 1024, "| csv KB:", os.path.getsize("silver_transactions.csv") // 1024)
q("SELECT month, count(*) AS n FROM 'silver_transactions.parquet' GROUP BY month ORDER BY month")

In [ ]:
# SQL result -> pandas DataFrame
df = con.sql("SELECT month, segment, sum(amount_gbp) AS spend FROM silver_transactions GROUP BY ALL").df()
print(type(df))
df.head(3)

In [ ]:
# pandas DataFrame -> SQL: DuckDB finds the variable by name
small = pd.DataFrame({"currency": ["EUR", "USD"], "label": ["Euro", "US dollar"]})
q("SELECT s.currency, s.label, count(*) AS n FROM silver_transactions AS t JOIN small AS s USING (currency) GROUP BY ALL")

In [ ]:
# Keep a database on disk (survives the session) by connecting to a file instead of memory:
#   con = duckdb.connect("card_spend.duckdb")
# Tables and views you create are then stored in that file.
for f in ["silver_transactions.csv", "silver_transactions.parquet"]:
    if os.path.exists(f):
        os.remove(f)

**Use when:** the end of a stage. Save with a clear name, record how many rows you wrote, and never overwrite the raw file.

## 17. Quick reference (pandas next to SQL)

In [ ]:
ref = pd.DataFrame([
    ("Read a CSV",               "pd.read_csv(path)",                          "SELECT * FROM read_csv('path')"),
    ("Look at data",             "df.head(), df.sample(5)",                    "SELECT * FROM t LIMIT 5; ... USING SAMPLE 5 ROWS"),
    ("Types and summary",        "df.info(), df.describe()",                   "DESCRIBE t; SUMMARIZE t"),
    ("Choose columns",           "df[['a','b']]",                              "SELECT a, b FROM t"),
    ("Drop / rename columns",    "drop(columns=...), rename(...)",             "SELECT * EXCLUDE (a); SELECT * RENAME (a AS b)"),
    ("Filter rows",              "df[df.x > 5], df.query(...)",                "WHERE x > 5"),
    ("In a list / range",        "isin([...]), between(a, b)",                 "x IN (...), x BETWEEN a AND b"),
    ("Change type",              "astype, to_numeric(errors='coerce')",        "CAST(x AS T), x::T, TRY_CAST(x AS T)"),
    ("Is missing",               "isna(), notna()",                            "x IS NULL, x IS NOT NULL"),
    ("Fill missing",             "fillna(v), ffill()",                         "coalesce(x, v), last_value(x IGNORE NULLS) OVER (...)"),
    ("Empty string to missing",  "replace('', np.nan)",                        "NULLIF(x, '')"),
    ("Duplicates",               "drop_duplicates()",                          "SELECT DISTINCT *"),
    ("Latest row per key",       "sort_values().groupby().tail(1)",            "QUALIFY row_number() OVER (PARTITION BY k ORDER BY d DESC) = 1"),
    ("Text cleaning",            "str.strip(), upper(), replace()",            "trim(x), upper(x), regexp_replace(x, p, r, 'g')"),
    ("Text search",              "str.contains(), str.startswith()",           "x LIKE '%a%', contains(x, 'a'), starts_with(x, 'a')"),
    ("If / else",                "np.where(c, a, b)",                          "CASE WHEN c THEN a ELSE b END, if(c, a, b)"),
    ("Several conditions",       "np.select(...)",                             "CASE WHEN ... WHEN ... ELSE ... END"),
    ("Bands",                    "pd.cut, pd.qcut",                            "CASE WHEN ..., ntile(n) OVER (ORDER BY x)"),
    ("Clip",                     "clip(lower, upper)",                         "least(greatest(x, lo), hi)"),
    ("Parse dates",              "pd.to_datetime(format=...)",                 "strptime / try_strptime(x, fmt)"),
    ("Date parts",               "dt.year, dt.month, dt.day_name()",           "year(x), month(x), dayname(x)"),
    ("Month key",                "dt.to_period('M')",                          "date_trunc('month', x), strftime(x, '%Y-%m')"),
    ("Date difference",          "(a - b).dt.days",                            "date_diff('day', b, a)"),
    ("Calendar of dates",        "pd.date_range",                              "generate_series(start, end, INTERVAL 1 DAY)"),
    ("Sort",                     "sort_values(['a','b'], ascending=[True, False])", "ORDER BY a, b DESC"),
    ("Top n",                    "nlargest(n, 'x')",                           "ORDER BY x DESC LIMIT n"),
    ("Rank",                     "rank(method='first')",                       "row_number() / rank() / dense_rank() OVER (...)"),
    ("Group and summarise",      "groupby().agg(...)",                         "GROUP BY ... with count/sum/avg/min/max"),
    ("Conditional aggregate",    "df[mask].x.sum()",                           "sum(x) FILTER (WHERE cond)"),
    ("Filter groups",            "groupby().filter(...)",                      "HAVING"),
    ("Group value on each row",  "groupby().transform('sum')",                 "sum(x) OVER (PARTITION BY k)"),
    ("Running calculations",     "shift, diff, cumsum, rolling",               "lag, lead, sum() OVER (ORDER BY ...), ROWS BETWEEN ..."),
    ("Wide / long",              "pivot_table, melt",                          "PIVOT ... ON ... USING ..., UNPIVOT ..."),
    ("List to rows",             "explode",                                    "unnest(list)"),
    ("Join",                     "merge(how=..., on=..., validate=...)",       "LEFT / INNER / FULL JOIN ... ON / USING (check key uniqueness yourself)"),
    ("Join: no match",           "merge(indicator=True)",                      "WHERE NOT EXISTS (...) or LEFT JOIN ... WHERE b.k IS NULL"),
    ("Nearest earlier row",      "merge_asof",                                 "ASOF JOIN ... ON a.k = b.k AND a.d >= b.d"),
    ("Stack tables",             "pd.concat",                                  "UNION ALL"),
    ("Set difference",           "isin / set operations",                      "EXCEPT, INTERSECT"),
    ("Quantiles",                "quantile(0.9)",                              "quantile_cont(x, 0.9), median(x)"),
    ("Checks",                   "assert df.x.is_unique",                      "a query that returns bad rows (should be empty)"),
    ("Pipeline",                 "pipe(), method chaining",                    "WITH step1 AS (...), step2 AS (...) SELECT ..."),
    ("Save",                     "to_csv, to_parquet",                         "COPY (SELECT ...) TO 'file' (...)"),
    ("Move between the two",     "con.sql(...).df()",                          "SELECT ... FROM my_dataframe"),
], columns=["Task", "pandas", "SQL (DuckDB)"])
pd.set_option("display.max_colwidth", None, "display.width", 250, "display.max_rows", 100)
ref.style.hide(axis="index").set_properties(**{"text-align": "left"})

### Habits worth keeping
- Look before you change: `DESCRIBE`, `SUMMARIZE`, and `GROUP BY` counts.
- Count rows before and after every step that can change the row count (joins, `DISTINCT`, `GROUP BY`).
- Convert types early; use `TRY_CAST`, then count what failed.
- Filter early, join late; name each step with a CTE and say in a comment what it assumes.
- Write every check as a query that returns the bad rows. Empty means pass.
- Keep the raw tables untouched. Build cleaned tables and views on top.

### pandas or SQL?
Both did the same work here. SQL is the natural choice when the data already lives in a warehouse, when the same logic must be run every day by a tool such as dbt, and when others need to read it. pandas is the natural choice for exploring, for steps that are awkward in SQL (custom Python functions, plotting, machine learning) and for small one-off jobs. DuckDB lets you use either on the same data.